# Phase 2: Heterogeneous Graph Attention Network (GAT) Summary  

## 1. Model Overview  

Unlike the previous homogeneous GNN, this model explicitly represents different entities in the venture capital ecosystem as distinct node types, enabling richer relational learning.

The model incorporates:
- **Company nodes** (prediction target)  
- **Person nodes** (founders and team members)  
- **Investor nodes** (venture capital firms and angels)  

**Why choose GAT over SHGMNN?**

GAT improves upon GraphSAGE by learning attention weights over neighboring nodes, allowing the model to assign different importance to different connections. This is particularly valuable in the venture capital setting, where certain investors and founders are significantly more influential than others. By focusing on high-quality relationships rather than treating all neighbors equally, GAT can capture more informative structural signals.

## 2. Graph Structure  

### Node Types  
- **Company**: startups with labeled outcomes (`is_success`)  
- **Person**: individuals associated with startups  
- **Investor**: entities investing in startups  

### Edge Types  
- **Person → Company (`works_on`)**  
  captures founder and team relationships  

- **Investor → Company (`invests_in`)**  
  captures funding relationships  

Reverse edges are included to allow bidirectional information flow during message passing.

## 3. Feature Design  

### Company Features  
- Tabular and network-derived features from Phase 1  
- Includes structural metrics (e.g., centrality) and firm-level attributes  
- Preprocessed using median imputation and standard scaling  

### Person Features  
- Role-based aggregates:
  - number of company affiliations  
  - founder, executive, and advisor counts  
- Profile indicators:
  - education availability  
  - LinkedIn presence  
  - gender encoding  

### Investor Features  
- Investment activity:
  - number of investments  
  - portfolio size within labeled dataset  
- Financial proxies:
  - average round size  
- Type indicators:
  - venture capital, angel, corporate VC  


## 4. Model Architecture  

The model uses a **2-layer heterogeneous GAT architecture**:

- Each edge type has its own attention-based message passing layer  
- Multi-head attention is used in the first layer to improve representation learning  
- Outputs from different relations are aggregated using summation  

At each layer, node representations are updated as a weighted combination of neighboring nodes, where the weights are learned through attention mechanisms.

Final prediction is performed using a linear classifier applied to **company node embeddings**.


## 5. Training Setup  
- Data split:
  - 64% training  
  - 16% validation  
  - 20% test  

# Phase 2 Extension: Incorporating Education and Job Information  

## 1. Objective  

This extension aims to evaluate whether **education and employment information** can improve model performance when incorporated into the graph structure, rather than treated as flat tabular features.  

To test this, we construct two heterogeneous GAT models with increasing levels of structural information.


## 2. Model Design  

### Model A: Baseline Heterogeneous GAT  
This model includes three node types:  
- **Company** (prediction target)  
- **Person** (founders and team members)  
- **Investor** (venture capital entities)  

Edges capture:
- person → company relationships  
- investor → company relationships  

### Model B: Extended Heterogeneous GAT  
This model extends Model A by introducing two additional node types:  
- **School** (education institutions)  
- **Organization** (past employers)  

New relationships include:
- person → school (education history)  
- person → organization (employment history)  

## 3. Key Difference  

The key difference between the two models lies in how education and job information are utilized:

- **Model A**: does not include education or job structure  
- **Model B**: integrates education and job data as **graph relationships**, enabling multi-hop information flow  

Instead of encoding attributes such as education level as static features, Model B represents them as **nodes in the graph**, allowing the model to learn structural patterns such as shared institutional backgrounds.

In [38]:
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv


In [39]:
# ============================================================
# 1. File paths
# ============================================================
FEATURE_PATH = "feature_matrix.csv"
EDU_JOB_PATH = "edu_job_features.csv"

COMPANY_TEAM_PATH = "company_team.csv"
PORTFOLIO_PATH = "portfolio_edges.csv"
INVESTORS_PATH = "investors.csv"
PEOPLE_PATH = "people.csv"

EDUCATION_PATH = "education.csv"
JOBS_PATH = "jobs.csv"


In [40]:
# ============================================================
# 2. Load data
# ============================================================
base = pd.read_csv(FEATURE_PATH)
edu_job = pd.read_csv(EDU_JOB_PATH)

company_team = pd.read_csv(COMPANY_TEAM_PATH)
portfolio = pd.read_csv(PORTFOLIO_PATH)
investors = pd.read_csv(INVESTORS_PATH)
people = pd.read_csv(PEOPLE_PATH)

education = pd.read_csv(EDUCATION_PATH)
jobs = pd.read_csv(JOBS_PATH)

print("Loaded shapes:")
print("feature_matrix:", base.shape)
print("edu_job_features:", edu_job.shape)
print("company_team:", company_team.shape)
print("portfolio:", portfolio.shape)
print("investors:", investors.shape)
print("people:", people.shape)
print("education:", education.shape)
print("jobs:", jobs.shape)


Loaded shapes:
feature_matrix: (6704, 37)
edu_job_features: (6704, 15)
company_team: (11102, 4)
portfolio: (461543, 6)
investors: (16864, 7)
people: (81498, 8)
education: (68963, 9)
jobs: (201788, 8)


In [41]:
# ============================================================
# 3. Merge base + edu_job for company features
# ============================================================
full = base.merge(edu_job, on="company_uuid", how="left")
full = full[full["is_success"].notna()].copy()
full["is_success"] = full["is_success"].astype(int)

print("\nLabeled companies:", full.shape)
print(full["is_success"].value_counts())



Labeled companies: (6593, 51)
is_success
0    5477
1    1116
Name: count, dtype: int64


In [42]:
# ============================================================
# 4. Feature definitions
# ============================================================
leakage_cols = [
    "funding_total_usd",
    "log_funding",
    "num_funding_rounds",
    "company_age_months",
]

id_target_cols = ["company_uuid", "is_success"]

# Company uses V2-style features by default
company_feature_cols = [
    c for c in full.columns
    if c not in leakage_cols + id_target_cols
]

print("\nCompany feature count:", len(company_feature_cols))



Company feature count: 45


In [43]:

# ============================================================
# 5. Helper: preprocess numeric features
# ============================================================
def preprocess_numeric_features(df, feature_cols):
    X = df[feature_cols].copy()

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    return X_scaled, imputer, scaler



In [44]:
# ============================================================
# 6. Company nodes
# ============================================================
company_df = full[["company_uuid", "is_success"] + company_feature_cols].copy()
company_ids = company_df["company_uuid"].tolist()
company_to_idx = {cid: i for i, cid in enumerate(company_ids)}

y = company_df["is_success"].values.astype(int)

X_company, company_imputer, company_scaler = preprocess_numeric_features(
    company_df, company_feature_cols
)

print("Num company nodes:", len(company_ids))
print("Company feature matrix:", X_company.shape)


Num company nodes: 6593
Company feature matrix: (6593, 45)


In [45]:
# ============================================================
# 7. Train / val / test masks on company nodes
# ============================================================
n_companies = len(company_ids)
all_idx = np.arange(n_companies)

train_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    stratify=y,
    random_state=42
)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.20,   # 64/16/20 overall
    stratify=y[train_idx],
    random_state=42
)

def make_mask(indices, n):
    mask = torch.zeros(n, dtype=torch.bool)
    mask[indices] = True
    return mask

train_mask = make_mask(train_idx, n_companies)
val_mask = make_mask(val_idx, n_companies)
test_mask = make_mask(test_idx, n_companies)

print("\nSplit sizes:")
print("Train:", train_mask.sum().item())
print("Val:", val_mask.sum().item())
print("Test:", test_mask.sum().item())



Split sizes:
Train: 4219
Val: 1055
Test: 1319


In [46]:
# ============================================================
# 8. Person nodes and features
# ============================================================
company_team = company_team[company_team["company_uuid"].isin(company_to_idx.keys())].copy()

# Start broad; later you can restrict to founder only
allowed_roles = {"founder", "c_suite", "board_member", "advisor", "vp", "other"}
company_team = company_team[company_team["role"].isin(allowed_roles)].copy()

person_ids = sorted(company_team["person_uuid"].dropna().unique().tolist())
person_to_idx = {pid: i for i, pid in enumerate(person_ids)}

people_sub = people[people["person_uuid"].isin(person_ids)].copy()

person_role_counts = (
    company_team.groupby("person_uuid")
    .agg(
        num_company_links=("company_uuid", "count"),
        founder_count=("role", lambda x: (x == "founder").sum()),
        c_suite_count=("role", lambda x: (x == "c_suite").sum()),
        board_count=("role", lambda x: (x == "board_member").sum()),
        advisor_count=("role", lambda x: (x == "advisor").sum()),
    )
    .reset_index()
)

people_sub = people_sub.merge(person_role_counts, on="person_uuid", how="left")

people_sub["education_fetched"] = people_sub["education_fetched"].fillna(0).astype(float)
people_sub["has_linkedin"] = people_sub["linkedin"].notna().astype(float)
people_sub["is_female"] = (people_sub["gender"] == "female").astype(float)
people_sub["is_male"] = (people_sub["gender"] == "male").astype(float)

person_feature_cols = [
    "education_fetched",
    "has_linkedin",
    "is_female",
    "is_male",
    "num_company_links",
    "founder_count",
    "c_suite_count",
    "board_count",
    "advisor_count",
]

for col in person_feature_cols:
    if col not in people_sub.columns:
        people_sub[col] = 0.0

people_sub = people_sub.set_index("person_uuid").reindex(person_ids).reset_index()

X_person, person_imputer, person_scaler = preprocess_numeric_features(
    people_sub, person_feature_cols
)

print("\nNum person nodes:", len(person_ids))
print("Person feature matrix:", X_person.shape)



Num person nodes: 11032
Person feature matrix: (11032, 9)


In [47]:
# ============================================================
# 9. Investor nodes and features
# ============================================================
portfolio = portfolio[portfolio["portfolio_company_uuid"].isin(company_to_idx.keys())].copy()

investor_ids = sorted(portfolio["vc_uuid"].dropna().unique().tolist())
investor_to_idx = {iid: i for i, iid in enumerate(investor_ids)}

investor_sub = investors[investors["investor_uuid"].isin(investor_ids)].copy()

investor_aggs = (
    portfolio.groupby("vc_uuid")
    .agg(
        labeled_portfolio_count=("portfolio_company_uuid", "nunique"),
        avg_round_size=("money_raised_usd", "mean"),
        recent_invest_count=("announced_on", "count"),
    )
    .reset_index()
    .rename(columns={"vc_uuid": "investor_uuid"})
)

investor_sub = investor_sub.merge(investor_aggs, on="investor_uuid", how="left")

investor_sub["investment_count"] = investor_sub["investment_count"].fillna(0)
investor_sub["has_website"] = investor_sub["website"].notna().astype(float)
investor_sub["is_vc"] = (investor_sub["investor_type"] == "venture_capital").astype(float)
investor_sub["is_angel"] = (investor_sub["investor_type"] == "angel").astype(float)
investor_sub["is_cvc"] = (investor_sub["investor_type"] == "corporate_venture_capital").astype(float)

investor_feature_cols = [
    "investment_count",
    "labeled_portfolio_count",
    "avg_round_size",
    "recent_invest_count",
    "has_website",
    "is_vc",
    "is_angel",
    "is_cvc",
]

for col in investor_feature_cols:
    if col not in investor_sub.columns:
        investor_sub[col] = 0.0

investor_sub = investor_sub.set_index("investor_uuid").reindex(investor_ids).reset_index()

X_investor, investor_imputer, investor_scaler = preprocess_numeric_features(
    investor_sub, investor_feature_cols
)

print("\nNum investor nodes:", len(investor_ids))
print("Investor feature matrix:", X_investor.shape)




Num investor nodes: 8504
Investor feature matrix: (8504, 8)


In [48]:
# ============================================================
# 10. Shared base edges
# ============================================================
# person -> company
pc_edges = company_team[
    company_team["person_uuid"].isin(person_to_idx.keys()) &
    company_team["company_uuid"].isin(company_to_idx.keys())
].copy()

person_src = pc_edges["person_uuid"].map(person_to_idx).to_numpy()
company_dst = pc_edges["company_uuid"].map(company_to_idx).to_numpy()

edge_person_company = torch.tensor([person_src, company_dst], dtype=torch.long)
edge_company_person = torch.tensor([company_dst, person_src], dtype=torch.long)

# investor -> company
ic_edges = portfolio[
    portfolio["vc_uuid"].isin(investor_to_idx.keys()) &
    portfolio["portfolio_company_uuid"].isin(company_to_idx.keys())
].copy()

investor_src = ic_edges["vc_uuid"].map(investor_to_idx).to_numpy()
company_dst2 = ic_edges["portfolio_company_uuid"].map(company_to_idx).to_numpy()

edge_investor_company = torch.tensor([investor_src, company_dst2], dtype=torch.long)
edge_company_investor = torch.tensor([company_dst2, investor_src], dtype=torch.long)

print("\nBase edges:")
print("person->company:", edge_person_company.shape[1])
print("investor->company:", edge_investor_company.shape[1])


Base edges:
person->company: 11032
investor->company: 38607


In [49]:
# ============================================================
# 11. Education nodes and edges
# ============================================================
education_sub = education[education["person_uuid"].isin(person_to_idx.keys())].copy()
education_sub = education_sub[education_sub["institution_name"].notna()].copy()

# Limit to most frequent schools to control graph size
top_school_count = 1000
top_schools = education_sub["institution_name"].value_counts().head(top_school_count).index.tolist()
education_sub = education_sub[education_sub["institution_name"].isin(top_schools)].copy()

school_ids = sorted(education_sub["institution_name"].unique().tolist())
school_to_idx = {sid: i for i, sid in enumerate(school_ids)}

school_degree = education_sub["institution_name"].value_counts().reindex(school_ids).fillna(0).values
X_school = np.log1p(school_degree).reshape(-1, 1)

# person -> school
edu_edges = education_sub[
    education_sub["person_uuid"].isin(person_to_idx.keys()) &
    education_sub["institution_name"].isin(school_to_idx.keys())
].copy()

edu_person_src = edu_edges["person_uuid"].map(person_to_idx).to_numpy()
edu_school_dst = edu_edges["institution_name"].map(school_to_idx).to_numpy()

edge_person_school = torch.tensor([edu_person_src, edu_school_dst], dtype=torch.long)
edge_school_person = torch.tensor([edu_school_dst, edu_person_src], dtype=torch.long)

print("\nNum school nodes:", len(school_ids))
print("person->school edges:", edge_person_school.shape[1])



Num school nodes: 1000
person->school edges: 12528


In [50]:
# ============================================================
# 12. Job organization nodes and edges
# ============================================================
jobs_sub = jobs[jobs["person_uuid"].isin(person_to_idx.keys())].copy()
jobs_sub = jobs_sub[jobs_sub["organization_name"].notna()].copy()

# Limit to most frequent organizations to control graph size
top_org_count = 2000
top_orgs = jobs_sub["organization_name"].value_counts().head(top_org_count).index.tolist()
jobs_sub = jobs_sub[jobs_sub["organization_name"].isin(top_orgs)].copy()

org_ids = sorted(jobs_sub["organization_name"].unique().tolist())
org_to_idx = {oid: i for i, oid in enumerate(org_ids)}

org_degree = jobs_sub["organization_name"].value_counts().reindex(org_ids).fillna(0).values
X_org = np.log1p(org_degree).reshape(-1, 1)

# person -> organization
job_edges = jobs_sub[
    jobs_sub["person_uuid"].isin(person_to_idx.keys()) &
    jobs_sub["organization_name"].isin(org_to_idx.keys())
].copy()

job_person_src = job_edges["person_uuid"].map(person_to_idx).to_numpy()
job_org_dst = job_edges["organization_name"].map(org_to_idx).to_numpy()

edge_person_org = torch.tensor([job_person_src, job_org_dst], dtype=torch.long)
edge_org_person = torch.tensor([job_org_dst, job_person_src], dtype=torch.long)

print("\nNum organization nodes:", len(org_ids))
print("person->organization edges:", edge_person_org.shape[1])



Num organization nodes: 2000
person->organization edges: 17418


In [51]:
# ============================================================
# 13. Build Model A data (without edu/job graph)
# ============================================================
def build_model_a_data():
    data = HeteroData()

    data["company"].x = torch.tensor(X_company, dtype=torch.float32)
    data["person"].x = torch.tensor(X_person, dtype=torch.float32)
    data["investor"].x = torch.tensor(X_investor, dtype=torch.float32)

    data["company"].y = torch.tensor(y, dtype=torch.long)
    data["company"].train_mask = train_mask
    data["company"].val_mask = val_mask
    data["company"].test_mask = test_mask

    data["person", "works_on", "company"].edge_index = edge_person_company
    data["company", "worked_by", "person"].edge_index = edge_company_person

    data["investor", "invests_in", "company"].edge_index = edge_investor_company
    data["company", "backed_by", "investor"].edge_index = edge_company_investor

    return data

In [52]:
# ============================================================
# 14. Build Model B data (with edu/job graph)
# ============================================================
def build_model_b_data():
    data = HeteroData()

    data["company"].x = torch.tensor(X_company, dtype=torch.float32)
    data["person"].x = torch.tensor(X_person, dtype=torch.float32)
    data["investor"].x = torch.tensor(X_investor, dtype=torch.float32)
    data["school"].x = torch.tensor(X_school, dtype=torch.float32)
    data["organization"].x = torch.tensor(X_org, dtype=torch.float32)

    data["company"].y = torch.tensor(y, dtype=torch.long)
    data["company"].train_mask = train_mask
    data["company"].val_mask = val_mask
    data["company"].test_mask = test_mask

    data["person", "works_on", "company"].edge_index = edge_person_company
    data["company", "worked_by", "person"].edge_index = edge_company_person

    data["investor", "invests_in", "company"].edge_index = edge_investor_company
    data["company", "backed_by", "investor"].edge_index = edge_company_investor

    data["person", "studied_at", "school"].edge_index = edge_person_school
    data["school", "has_student", "person"].edge_index = edge_school_person

    data["person", "worked_at", "organization"].edge_index = edge_person_org
    data["organization", "employs", "person"].edge_index = edge_org_person

    return data


data_a = build_model_a_data()
data_b = build_model_b_data()

print("\nModel A data:")
print(data_a)

print("\nModel B data:")
print(data_b)


Model A data:
HeteroData(
  company={
    x=[6593, 45],
    y=[6593],
    train_mask=[6593],
    val_mask=[6593],
    test_mask=[6593],
  },
  person={ x=[11032, 9] },
  investor={ x=[8504, 8] },
  (person, works_on, company)={ edge_index=[2, 11032] },
  (company, worked_by, person)={ edge_index=[2, 11032] },
  (investor, invests_in, company)={ edge_index=[2, 38607] },
  (company, backed_by, investor)={ edge_index=[2, 38607] }
)

Model B data:
HeteroData(
  company={
    x=[6593, 45],
    y=[6593],
    train_mask=[6593],
    val_mask=[6593],
    test_mask=[6593],
  },
  person={ x=[11032, 9] },
  investor={ x=[8504, 8] },
  school={ x=[1000, 1] },
  organization={ x=[2000, 1] },
  (person, works_on, company)={ edge_index=[2, 11032] },
  (company, worked_by, person)={ edge_index=[2, 11032] },
  (investor, invests_in, company)={ edge_index=[2, 38607] },
  (company, backed_by, investor)={ edge_index=[2, 38607] },
  (person, studied_at, school)={ edge_index=[2, 12528] },
  (school, has_st

In [53]:
# ============================================================
# 15. Heterogeneous GAT model
# ============================================================
class FlexibleHeteroGAT(nn.Module):
    def __init__(self, relation_types, hidden_channels=64, out_channels=2, heads=2, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        conv1_dict = {}
        conv2_dict = {}

        for rel in relation_types:
            conv1_dict[rel] = GATConv(
                (-1, -1),
                hidden_channels,
                heads=heads,
                add_self_loops=False,
                dropout=dropout
            )
            conv2_dict[rel] = GATConv(
                (-1, -1),
                hidden_channels,
                heads=1,
                add_self_loops=False,
                dropout=dropout
            )

        self.conv1 = HeteroConv(conv1_dict, aggr="sum")
        self.conv2 = HeteroConv(conv2_dict, aggr="sum")

        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {k: F.elu(v) for k, v in x_dict.items()}
        x_dict = {k: F.dropout(v, p=self.dropout, training=self.training) for k, v in x_dict.items()}

        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {k: F.elu(v) for k, v in x_dict.items()}
        x_dict = {k: F.dropout(v, p=self.dropout, training=self.training) for k, v in x_dict.items()}

        out = self.classifier(x_dict["company"])
        return out


In [54]:
# ============================================================
# 16. Evaluation helper
# ============================================================
def evaluate_company_split(logits, y_true, mask, threshold=0.5):
    probs = F.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
    y_np = y_true.detach().cpu().numpy()
    mask_np = mask.detach().cpu().numpy()

    y_eval = y_np[mask_np]
    p_eval = probs[mask_np]
    pred_eval = (p_eval >= threshold).astype(int)

    roc = roc_auc_score(y_eval, p_eval)
    pr = average_precision_score(y_eval, p_eval)

    return {
        "roc_auc": roc,
        "pr_auc": pr,
        "y_true": y_eval,
        "y_pred": pred_eval,
        "y_prob": p_eval
    }

In [55]:

# ============================================================
# 17. Train function
# ============================================================
def train_hetero_gat(
    data,
    model_name="Model",
    hidden_channels=64,
    heads=2,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    patience=30,
    threshold=0.5
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = data.to(device)

    model = FlexibleHeteroGAT(
        relation_types=list(data.edge_index_dict.keys()),
        hidden_channels=hidden_channels,
        out_channels=2,
        heads=heads,
        dropout=0.3
    ).to(device)

    y_train = data["company"].y[data["company"].train_mask]
    num_pos = (y_train == 1).sum().item()
    num_neg = (y_train == 0).sum().item()

    class_weights = torch.tensor(
        [1.0, num_neg / max(num_pos, 1)],
        dtype=torch.float32,
        device=device
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val_roc = -np.inf
    wait = 0

    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(data.x_dict, data.edge_index_dict)
        loss = criterion(
            logits[data["company"].train_mask],
            data["company"].y[data["company"].train_mask]
        )

        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x_dict, data.edge_index_dict)
            val_metrics = evaluate_company_split(
                logits,
                data["company"].y,
                data["company"].val_mask,
                threshold=threshold
            )
            val_roc = val_metrics["roc_auc"]

        if val_roc > best_val_roc:
            best_val_roc = val_roc
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 20 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss {loss.item():.4f} | "
                f"Val ROC-AUC {val_metrics['roc_auc']:.4f} | "
                f"Val PR-AUC {val_metrics['pr_auc']:.4f}"
            )

        if wait >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        logits = model(data.x_dict, data.edge_index_dict)

    train_metrics = evaluate_company_split(
        logits, data["company"].y, data["company"].train_mask, threshold=threshold
    )
    val_metrics = evaluate_company_split(
        logits, data["company"].y, data["company"].val_mask, threshold=threshold
    )
    test_metrics = evaluate_company_split(
        logits, data["company"].y, data["company"].test_mask, threshold=threshold
    )

    print("\nBest model performance:")
    print(f"Train ROC-AUC: {train_metrics['roc_auc']:.4f} | Train PR-AUC: {train_metrics['pr_auc']:.4f}")
    print(f"Val   ROC-AUC: {val_metrics['roc_auc']:.4f} | Val   PR-AUC: {val_metrics['pr_auc']:.4f}")
    print(f"Test  ROC-AUC: {test_metrics['roc_auc']:.4f} | Test  PR-AUC: {test_metrics['pr_auc']:.4f}")

    print("\nTest classification report:")
    print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], digits=4))

    return {
        "model": model,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics
    }

In [56]:
# ============================================================
# 18. Run both models
# ============================================================
results_a = train_hetero_gat(
    data=data_a,
    model_name="Model A: company + person + investor",
    hidden_channels=64,
    heads=2,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    patience=30,
    threshold=0.5
)

results_b = train_hetero_gat(
    data=data_b,
    model_name="Model B: + school + organization",
    hidden_channels=64,
    heads=2,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    patience=30,
    threshold=0.5
)


Training Model A: company + person + investor
Epoch 001 | Loss 0.7355 | Val ROC-AUC 0.6642 | Val PR-AUC 0.3612
Epoch 020 | Loss 0.5730 | Val ROC-AUC 0.8023 | Val PR-AUC 0.5355
Epoch 040 | Loss 0.5411 | Val ROC-AUC 0.8204 | Val PR-AUC 0.5654
Epoch 060 | Loss 0.5235 | Val ROC-AUC 0.8305 | Val PR-AUC 0.5895
Epoch 080 | Loss 0.5112 | Val ROC-AUC 0.8338 | Val PR-AUC 0.5979
Epoch 100 | Loss 0.5045 | Val ROC-AUC 0.8336 | Val PR-AUC 0.5965
Epoch 120 | Loss 0.5009 | Val ROC-AUC 0.8362 | Val PR-AUC 0.6073
Epoch 140 | Loss 0.4924 | Val ROC-AUC 0.8338 | Val PR-AUC 0.6058
Epoch 160 | Loss 0.4741 | Val ROC-AUC 0.8354 | Val PR-AUC 0.6113
Early stopping at epoch 178

Best model performance:
Train ROC-AUC: 0.8799 | Train PR-AUC: 0.6591
Val   ROC-AUC: 0.8370 | Val   PR-AUC: 0.6062
Test  ROC-AUC: 0.7973 | Test  PR-AUC: 0.5171

Test classification report:
              precision    recall  f1-score   support

           0     0.9204    0.7810    0.8450      1096
           1     0.3830    0.6682    0.486

In [58]:
# ============================================================
# 19. Final comparison table
# ============================================================
summary = pd.DataFrame({
    "Model": [
        "A: company+person+investor",
        "B: +school+organization"
    ],
    "Val ROC-AUC": [
        results_a["val_metrics"]["roc_auc"],
        results_b["val_metrics"]["roc_auc"]
    ],
    "Val PR-AUC": [
        results_a["val_metrics"]["pr_auc"],
        results_b["val_metrics"]["pr_auc"]
    ],
    "Test ROC-AUC": [
        results_a["test_metrics"]["roc_auc"],
        results_b["test_metrics"]["roc_auc"]
    ],
    "Test PR-AUC": [
        results_a["test_metrics"]["pr_auc"],
        results_b["test_metrics"]["pr_auc"]
    ],
})

print("\nFinal comparison:")
print(summary)


Final comparison:
                        Model  Val ROC-AUC  Val PR-AUC  Test ROC-AUC  \
0  A: company+person+investor     0.837029    0.606194      0.797323   
1     B: +school+organization     0.842188    0.626706      0.788644   

   Test PR-AUC  
0     0.517090  
1     0.512366  
